In [3]:
df = pd.read_csv(r"C:\Users\Aman Kumar Singh\Desktop\nlp_dissertation\Distillation\sft_qwen25_15b\kbg_sft_dataset_1.csv")

In [13]:
import re
import pandas as pd

consultations = (
    df["consultation"]
    .dropna()
    .astype(str)
    .head(500)
    .tolist()
)

def extract_parent_responses(consultation):
    responses = re.findall(
        r"Parent:\s*(.*?)(?=\n\s*\nDoctor:|\Z)",
        consultation,
        flags=re.DOTALL | re.IGNORECASE
    )

    substantive_responses = []

    for response in responses:
        response = re.sub(r"\s+", " ", response).strip()
        normalised = response.lower().strip(" .,:;!?")

        if normalised in {
            "n/a", "na", "n.a.", "n.a",
            "none", "no", "not applicable"
        }:
            continue

        if response:
            substantive_responses.append(response)

    return substantive_responses

parent_consultations = [
    extract_parent_responses(consultation)
    for consultation in consultations
]

parent_consultation_text = [
    " ".join(responses)
    for responses in parent_consultations
]

num_responses = [
    len(responses)
    for responses in parent_consultations
]

print(f"Consultations: {len(parent_consultation_text)}")
print(f"Mean substantive responses: {sum(num_responses) / len(num_responses):.2f}")
print(f"Minimum substantive responses: {min(num_responses)}")
print(f"Median substantive responses: {pd.Series(num_responses).median():.0f}")
print(f"Maximum substantive responses: {max(num_responses)}")

Consultations: 500
Mean substantive responses: 7.00
Minimum substantive responses: 7
Median substantive responses: 7
Maximum substantive responses: 7


In [14]:
import numpy as np
import re

def calculate_mtld(tokens, threshold=0.72):
    if len(tokens) < 2:
        return np.nan

    factors = 0
    token_count = 0
    types = set()

    for token in tokens:
        token_count += 1
        types.add(token)

        ttr = len(types) / token_count

        if ttr <= threshold:
            factors += 1
            token_count = 0
            types = set()

    if token_count > 0:
        ttr = len(types) / token_count

        if ttr != 1:
            factors += (1 - ttr) / (1 - threshold)

    if factors == 0:
        return np.nan

    return len(tokens) / factors


parent_tokens = [
    re.findall(r"\b[a-zA-Z]+\b", consultation.lower())
    for consultation in parent_consultation_text
]

mtld_values = np.array([
    calculate_mtld(tokens)
    for tokens in parent_tokens
])

mtld_values = mtld_values[~np.isnan(mtld_values)]

print(f"Consultations: {len(mtld_values)}")
print(f"Mean MTLD: {mtld_values.mean():.2f}")
print(f"SD: {mtld_values.std(ddof=1):.2f}")
print(f"Minimum MTLD: {mtld_values.min():.2f}")
print(f"Median MTLD: {np.median(mtld_values):.2f}")
print(f"Maximum MTLD: {mtld_values.max():.2f}")

Consultations: 500
Mean MTLD: 130.12
SD: 36.06
Minimum MTLD: 51.75
Median MTLD: 126.01
Maximum MTLD: 252.71


In [7]:
!pip install lexicalrichness

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 625.0/625.0 kB 11.7 MB/s  0:00:00
  Created wheel for lexicalrichness: filename=lexicalrichness-0.5.1-py3-none-any.whl size=15565 sha256=967b7f7ac309006718c725dfba449ba191e7223361bc5e538fcc38ee38031839
  Stored in directory: c:\users\aman kumar singh\appdata\local\pip\cache\wheels\bb\68\f4\1a32ae0aae29ce426b4e3c106a4e8e9c19fd13bcaff45c6a8e
Successfully built lexicalrichness

   ---------------------------------------- 2/2 [lexicalrichness]



In [16]:
from lexicalrichness import LexicalRichness
import numpy as np

mtld_values_standard = []

for consultation in parent_consultation_text:
    tokens = consultation.lower().split()

    tokens = [
        token.strip(".,!?;:\"'()[]{}")
        for token in tokens
    ]

    tokens = [
        token
        for token in tokens
        if token.isalpha()
    ]

    if len(tokens) >= 20:
        text = " ".join(tokens)
        lex = LexicalRichness(text)
        mtld = lex.mtld(threshold=0.72)
        mtld_values_standard.append(mtld)

mtld_values_standard = np.array(mtld_values_standard)

print(f"Consultations: {len(mtld_values_standard)}")
print(f"Mean MTLD: {mtld_values_standard.mean():.2f}")
print(f"SD: {mtld_values_standard.std(ddof=1):.2f}")
print(f"Minimum MTLD: {mtld_values_standard.min():.2f}")
print(f"Median MTLD: {np.median(mtld_values_standard):.2f}")
print(f"Maximum MTLD: {mtld_values_standard.max():.2f}")

Consultations: 500
Mean MTLD: 143.90
SD: 36.17
Minimum MTLD: 42.35
Median MTLD: 137.15
Maximum MTLD: 272.79


In [15]:
import re
import numpy as np

consultation_mlu = []
all_parent_utterance_lengths = []

for responses in parent_consultations:
    utterance_lengths = []

    for response in responses:
        words = re.findall(
            r"\b[a-zA-Z]+(?:'[a-zA-Z]+)?\b",
            response
        )

        if words:
            utterance_lengths.append(len(words))

    if utterance_lengths:
        consultation_mlu.append(np.mean(utterance_lengths))
        all_parent_utterance_lengths.extend(utterance_lengths)

consultation_mlu = np.array(consultation_mlu)

responses_per_consultation = np.array([
    len(responses)
    for responses in parent_consultations
])

print(f"Consultations: {len(consultation_mlu)}")
print(f"Total parent utterances: {len(all_parent_utterance_lengths)}")
print(f"Mean MLU: {consultation_mlu.mean():.2f} words")
print(f"SD: {consultation_mlu.std(ddof=1):.2f}")
print(f"Minimum MLU: {consultation_mlu.min():.2f}")
print(f"Median MLU: {np.median(consultation_mlu):.2f}")
print(f"Maximum MLU: {consultation_mlu.max():.2f}")
print(f"Mean substantive responses: {responses_per_consultation.mean():.2f}")
print(f"Minimum substantive responses: {responses_per_consultation.min()}")
print(f"Median substantive responses: {np.median(responses_per_consultation):.0f}")
print(f"Maximum substantive responses: {responses_per_consultation.max()}")

Consultations: 500
Total parent utterances: 3500
Mean MLU: 26.70 words
SD: 8.36
Minimum MLU: 6.57
Median MLU: 27.00
Maximum MLU: 60.71
Mean substantive responses: 7.00
Minimum substantive responses: 7
Median substantive responses: 7
Maximum substantive responses: 7


In [ ]:
import nltk
import numpy as np
from collections import Counter

nltk.download("averaged_perceptron_tagger_eng")

penn_treebank_tags = {
    "CC", "CD", "DT", "EX", "FW", "IN",
    "JJ", "JJR", "JJS",
    "LS", "MD",
    "NN", "NNS", "NNP", "NNPS",
    "PDT", "POS",
    "PRP", "PRP$",
    "RB", "RBR", "RBS", "RP",
    "SYM", "TO", "UH",
    "VB", "VBD", "VBG", "VBN", "VBP", "VBZ",
    "WDT", "WP", "WP$", "WRB"
}

all_parent_tokens = []

for responses in parent_consultations:
    for response in responses:
        tokens = [
            token.strip(".,!?;:\"'()[]{}")
            for token in response.lower().split()
        ]

        tokens = [
            token
            for token in tokens
            if token.isalpha()
        ]

        all_parent_tokens.extend(tokens)

tagged_tokens = nltk.pos_tag(all_parent_tokens)

all_pos_tags = [
    tag
    for _, tag in tagged_tokens
    if tag in penn_treebank_tags
]

pos_counts = Counter(all_pos_tags)

total_pos = sum(pos_counts.values())

pos_probabilities = np.array([
    count / total_pos
    for count in pos_counts.values()
])

observed_entropy = -np.sum(
    pos_probabilities * np.log2(pos_probabilities)
)

maximum_entropy = np.log2(len(penn_treebank_tags))

normalised_entropy = observed_entropy / maximum_entropy

print(f"Total parent word tokens: {len(all_parent_tokens)}")
print(f"Observed POS categories: {len(pos_counts)}")
print(f"Observed entropy: {observed_entropy:.4f}")
print(f"Maximum entropy: {maximum_entropy:.4f}")
print(f"Normalised POS entropy: {normalised_entropy:.4f}")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Aman Kumar Singh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Total parent word tokens: 87630
Observed POS categories: 30
Observed entropy: 4.1323
Maximum entropy: 5.1699
Normalised POS entropy: 0.7993


In [18]:
import nltk
import numpy as np
from collections import Counter

nltk.download("averaged_perceptron_tagger_eng")

penn_treebank_tags = {
    "CC", "CD", "DT", "EX", "FW", "IN",
    "JJ", "JJR", "JJS",
    "LS", "MD",
    "NN", "NNS", "NNP", "NNPS",
    "PDT", "POS",
    "PRP", "PRP$",
    "RB", "RBR", "RBS", "RP",
    "SYM", "TO", "UH",
    "VB", "VBD", "VBG", "VBN", "VBP", "VBZ",
    "WDT", "WP", "WP$", "WRB"
}

all_parent_tokens = []

for responses in parent_consultations:
    for response in responses:
        tokens = response.lower().split()

        tokens = [
            token.strip(".,!?;:\"'()[]{}")
            for token in tokens
        ]

        tokens = [
            token
            for token in tokens
            if token.isalpha()
        ]

        all_parent_tokens.extend(tokens)

tagged_tokens = nltk.pos_tag(all_parent_tokens)

all_pos_tags = [
    tag
    for _, tag in tagged_tokens
    if tag in penn_treebank_tags
]

pos_counts = Counter(all_pos_tags)

total_pos = sum(pos_counts.values())

pos_probabilities = np.array([
    count / total_pos
    for count in pos_counts.values()
])

observed_entropy = -np.sum(
    pos_probabilities * np.log2(pos_probabilities)
)

num_possible_tags = len(penn_treebank_tags)

maximum_entropy = np.log2(num_possible_tags)

normalised_entropy = observed_entropy / maximum_entropy

print(f"Total parent word tokens: {len(all_parent_tokens)}")
print(f"Observed POS categories: {len(pos_counts)}")
print(f"Observed entropy: {observed_entropy:.4f}")
print(f"Maximum entropy: {maximum_entropy:.4f}")
print(f"Normalised POS entropy: {normalised_entropy:.4f}")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Aman Kumar Singh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Total parent word tokens: 87630
Observed POS categories: 30
Observed entropy: 4.1323
Maximum entropy: 5.1699
Normalised POS entropy: 0.7993


In [ ]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("=" * 60)
print("SEMANTIC SIMILARITY — PARENT RESPONSES ONLY")
print("=" * 60)

print(f"Consultations analysed : {len(parent_consultation_text)}")

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)



embeddings = model.encode(
    parent_consultation_text,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"Embedding shape         : {embeddings.shape}")


similarity_matrix = cosine_similarity(embeddings)

n = len(parent_consultation_text)

pairwise_similarities = similarity_matrix[
    np.triu_indices(n, k=1)
]
mean_similarity = pairwise_similarities.mean()
sd_similarity = pairwise_similarities.std(ddof=1)
min_similarity = pairwise_similarities.min()
median_similarity = np.median(pairwise_similarities)
max_similarity = pairwise_similarities.max()

print("\n" + "=" * 60)
print("COSINE SIMILARITY RESULTS")
print("=" * 60)

print(f"Number of unique pairs : {len(pairwise_similarities)}")

print(f"Mean similarity        : {mean_similarity:.4f}")
print(f"SD                     : {sd_similarity:.4f}")
print(f"Minimum similarity     : {min_similarity:.4f}")
print(f"Median similarity      : {median_similarity:.4f}")
print(f"Maximum similarity     : {max_similarity:.4f}")


semantic_diversity = 1 - mean_similarity

print("\n" + "=" * 60)
print("SEMANTIC DIVERSITY")
print("=" * 60)

print(f"1 - mean cosine similarity : {semantic_diversity:.4f}")


SEMANTIC SIMILARITY — PARENT RESPONSES ONLY
Consultations analysed : 500


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding shape         : (500, 384)

COSINE SIMILARITY RESULTS
Number of unique pairs : 124750
Mean similarity        : 0.6505
SD                     : 0.0811
Minimum similarity     : 0.2956
Median similarity      : 0.6521
Maximum similarity     : 0.9241

SEMANTIC DIVERSITY
1 - mean cosine similarity : 0.3495
